# Restaurant ETA Prediction System

## Objective

Build a scikit-learn regression pipeline to predict delivery/fulfillment time for restaurant orders in minutes.

In this project I've included:

- data inspection and cleaning
- feature engineering
- leakage-safe preprocessing using `Pipeline` and `ColumnTransformer`
- comparison of multiple regression algorithms
- hyperparameter tuning
- evaluation using MAE, RMSE, and R²
- error analysis

### Prediction-time assumption

The ETA is assumed to be generated when the customer places the order. Therefore, information that becomes available only after order placement, such as `Time_Order_picked`, has been excluded to avoid temporal leakage.

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("train.csv")

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (45593, 20)


,ID,Delivery_person_ID,Delivery_person_Age,Delivery_person_Ratings,Restaurant_latitude,Restaurant_longitude,Delivery_location_latitude,Delivery_location_longitude,Order_Date,Time_Orderd,Time_Order_picked,Weatherconditions,Road_traffic_density,Vehicle_condition,Type_of_order,Type_of_vehicle,multiple_deliveries,Festival,City,Time_taken(min)
0,0x4607,INDORES13DEL02,37,4.9,22.745049,75.892471,22.765049,75.912471,19-03-2022,11:30:00,11:45:00,conditions Sunny,High,2,Snack,motorcycle,0,No,Urban,(min) 24
1,0xb379,BANGRES18DEL02,34,4.5,12.913041,77.683237,13.043041,77.813237,25-03-2022,19:45:00,19:50:00,conditions Stormy,Jam,2,Snack,scooter,1,No,Metropolitian,(min) 33
2,0x5d6d,BANGRES19DEL01,23,4.4,12.914264,77.678400,12.924264,77.688400,19-03-2022,08:30:00,08:45:00,conditions Sandstorms,Low,0,Drinks,motorcycle,1,No,Urban,(min) 26
3,0x7a6a,COIMBRES13DEL02,38,4.7,11.003669,76.976494,11.053669,77.026494,05-04-2022,18:00:00,18:10:00,conditions Sunny,Medium,0,Buffet,motorcycle,1,No,Metropolitian,(min) 21
4,0x70a2,CHENRES12DEL01,32,4.6,12.972793,80.249982,13.012793,80.289982,26-03-2022,13:30:00,13:45:00,conditions Cloudy,High,1,Snack,scooter,1,No,Metropolitian,(min) 30


## Initial Data Inspection

Before preprocessing or modelling, I'll inspect the raw dataset to understand its structure, data types, missing-value representation, target format, and potential data-quality issues.

In [2]:
print("Shape:", df.shape)

print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

Shape: (45593, 20)

Column names:
['ID', 'Delivery_person_ID', 'Delivery_person_Age', 'Delivery_person_Ratings', 'Restaurant_latitude', 'Restaurant_longitude', 'Delivery_location_latitude', 'Delivery_location_longitude', 'Order_Date', 'Time_Orderd', 'Time_Order_picked', 'Weatherconditions', 'Road_traffic_density', 'Vehicle_condition', 'Type_of_order', 'Type_of_vehicle', 'multiple_deliveries', 'Festival', 'City', 'Time_taken(min)']

Data types:
ID                                 str
Delivery_person_ID                 str
Delivery_person_Age                str
Delivery_person_Ratings            str
Restaurant_latitude            float64
Restaurant_longitude           float64
Delivery_location_latitude     float64
Delivery_location_longitude    float64
Order_Date                         str
Time_Orderd                        str
Time_Order_picked                  str
Weatherconditions                  str
Road_traffic_density               str
Vehicle_condition                int64
Type_o

In [3]:
object_columns = df.select_dtypes(include="object").columns

for col in object_columns:
    print(f"\n--- {col} ---")
    print(df[col].value_counts(dropna=False).head(10))


--- ID ---
ID
0x4607     1
0xb379     1
0x5d6d     1
0x7a6a     1
0x70a2     1
0x9bb4     1
0x95b4     1
0x9eb2     1
0x1102     1
0xcdcd     1
Name: count, dtype: int64

--- Delivery_person_ID ---
Delivery_person_ID
JAPRES11DEL02        67
PUNERES01DEL01       67
RANCHIRES02DEL01     66
VADRES11DEL02        66
JAPRES03DEL01        66
HYDRES04DEL02        66
VADRES08DEL02        66
SURRES11DEL01        65
INDORES08DEL02       65
VADRES11DEL01        65
Name: count, dtype: int64

--- Delivery_person_Age ---
Delivery_person_Age
35    2262
36    2260
37    2227
30    2226
38    2219
24    2210
32    2202
22    2196
29    2191
33    2187
Name: count, dtype: int64

--- Delivery_person_Ratings ---
Delivery_person_Ratings
4.8     7148
4.7     7142
4.9     7041
4.6     6940
5       3996
4.5     3303
NaN     1908
4.1     1430
4.2     1418
4.3     1409
Name: count, dtype: int64

--- Order_Date ---
Order_Date
15-03-2022    1192
03-04-2022    1178
13-03-2022    1169
26-03-2022    1166
24-03-2022 

/var/folders/9v/fzccp8ks6mx74kxsphtk3_x80000gn/T/ipykernel_22919/3560094363.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  object_columns = df.select_dtypes(include="object").columns


In [4]:
print(df["Time_taken(min)"].head(10))
print("\nTarget dtype:", df["Time_taken(min)"].dtype)

0    (min) 24
1    (min) 33
2    (min) 26
3    (min) 21
4    (min) 30
5    (min) 26
6    (min) 40
7    (min) 32
8    (min) 34
9    (min) 46
Name: Time_taken(min), dtype: str

Target dtype: str


## Data Cleaning and Feature Engineering

The raw dataset contains numerical values stored as strings and text representations of missing values. These will be cleaned before modelling.

The target column `Time_taken(min)` is converted from strings like `(min) 24` into numerical delivery times like `24`.

In [5]:
df["Time_taken(min)"] = (
    df["Time_taken(min)"]
    .str.replace("(min)", "", regex=False)
    .str.strip()
    .astype(float)
)

In [6]:
print(df["Time_taken(min)"].head())
print("\nTarget dtype:", df["Time_taken(min)"].dtype)

print("\nTarget summary:")
print(df["Time_taken(min)"].describe())

0    24.0
1    33.0
2    26.0
3    21.0
4    30.0
Name: Time_taken(min), dtype: float64

Target dtype: float64

Target summary:
count    45593.000000
mean        26.294607
std          9.383806
min         10.000000
25%         19.000000
50%         26.000000
75%         32.000000
max         54.000000
Name: Time_taken(min), dtype: float64


In [7]:
X = df.drop(columns="Time_taken(min)").copy()
y = df["Time_taken(min)"].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (45593, 19)
y shape: (45593,)


In [8]:
string_columns = X.select_dtypes(include="str").columns

for col in string_columns:
    X[col] = X[col].str.strip()

X[string_columns] = X[string_columns].replace("NaN", np.nan)

In [9]:
X["Weatherconditions"] = (
    X["Weatherconditions"]
    .str.replace("conditions ", "", regex=False)
    .replace("NaN", np.nan)
)

In [10]:
print(X["Weatherconditions"].value_counts(dropna=False))

Weatherconditions
Fog           7654
Stormy        7586
Cloudy        7536
Sandstorms    7495
Windy         7422
Sunny         7284
NaN            616
Name: count, dtype: int64


In [11]:
numeric_string_columns = [
    "Delivery_person_Age",
    "Delivery_person_Ratings",
    "multiple_deliveries"
]

for col in numeric_string_columns:
    X[col] = pd.to_numeric(X[col], errors="coerce")

In [12]:
print(X[numeric_string_columns].dtypes)

print("\nMissing values:")
print(X[numeric_string_columns].isna().sum())

print("\nSummary:")
print(X[numeric_string_columns].describe())

Delivery_person_Age        float64
Delivery_person_Ratings    float64
multiple_deliveries        float64
dtype: object

Missing values:
Delivery_person_Age        1854
Delivery_person_Ratings    1908
multiple_deliveries         993
dtype: int64

Summary:
       Delivery_person_Age  Delivery_person_Ratings  multiple_deliveries
count         43739.000000             43685.000000         44600.000000
mean             29.567137                 4.633780             0.744664
std               5.815155                 0.334716             0.572473
min              15.000000                 1.000000             0.000000
25%              25.000000                 4.500000             0.000000
50%              30.000000                 4.700000             1.000000
75%              35.000000                 4.900000             1.000000
max              50.000000                 6.000000             3.000000


### Distance Feature Engineering

The four latitude/longitude columns are converted into a more directly useful feature, `distance_km`, which represents the straight-line geographical distance between the restaurant and delivery location.

The Haversine formula is used to account for the curvature of the Earth.

In [13]:
def haversine_distance(lat1, lon1, lat2, lon2):
    earth_radius_km = 6371.0

    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)

    delta_lat = lat2 - lat1
    delta_lon = lon2 - lon1

    a = (
        np.sin(delta_lat / 2) ** 2
        + np.cos(lat1)
        * np.cos(lat2)
        * np.sin(delta_lon / 2) ** 2
    )

    c = 2 * np.arcsin(np.sqrt(a))

    return earth_radius_km * c

In [14]:
X["distance_km"] = haversine_distance(
    X["Restaurant_latitude"],
    X["Restaurant_longitude"],
    X["Delivery_location_latitude"],
    X["Delivery_location_longitude"]
)

In [15]:
print(X["distance_km"].describe())

count    45593.000000
mean        99.303911
std       1099.731281
min          1.465067
25%          4.663493
50%          9.264281
75%         13.763977
max      19692.674606
Name: distance_km, dtype: float64


In [16]:
zero_restaurant_coords = (
    (X["Restaurant_latitude"] == 0)
    & (X["Restaurant_longitude"] == 0)
)

print(
    "Rows with restaurant coordinates (0, 0):",
    zero_restaurant_coords.sum()
)

print(
    "Percentage:",
    round(zero_restaurant_coords.mean() * 100, 2),
    "%"
)

print("\nLargest calculated distances:")
print(
    X[
        [
            "Restaurant_latitude",
            "Restaurant_longitude",
            "Delivery_location_latitude",
            "Delivery_location_longitude",
            "distance_km"
        ]
    ]
    .sort_values("distance_km", ascending=False)
    .head(10)
)

Rows with restaurant coordinates (0, 0): 3640
Percentage: 7.98 %

Largest calculated distances:
       Restaurant_latitude  Restaurant_longitude  Delivery_location_latitude  \
33533           -22.533662            -88.366217                   22.663662   
6788            -22.552672            -88.352885                   22.662672   
2484            -22.538999            -88.322337                   22.668999   
18826           -22.537960            -88.349843                   22.547960   
9535            -23.374989            -85.335486                   23.444989   
762             -23.359194            -85.325447                   23.449194   
35535           -23.369746            -85.339820                   23.419746   
22291           -23.351058            -85.325731                   23.421058   
30633           -23.357804            -85.325146                   23.417804   
43454           -23.351058            -85.325731                   23.401058   

       Delivery_locatio

In [17]:
order_time = pd.to_datetime(
    X["Time_Orderd"],
    format="%H:%M:%S",
    errors="coerce"
)

X["order_hour"] = (
    order_time.dt.hour
    + order_time.dt.minute / 60
)

In [18]:
print(X[["Time_Orderd", "order_hour"]].head(10))

  Time_Orderd  order_hour
0    11:30:00   11.500000
1    19:45:00   19.750000
2    08:30:00    8.500000
3    18:00:00   18.000000
4    13:30:00   13.500000
5    21:20:00   21.333333
6    19:15:00   19.250000
7    17:25:00   17.416667
8    20:55:00   20.916667
9    21:55:00   21.916667


In [19]:
order_date = pd.to_datetime(
    X["Order_Date"],
    format="%d-%m-%Y",
    errors="coerce"
)

X["day_of_week"] = order_date.dt.day_name()

In [20]:
print(X[["Order_Date", "day_of_week"]].head())

   Order_Date day_of_week
0  19-03-2022    Saturday
1  25-03-2022      Friday
2  19-03-2022    Saturday
3  05-04-2022     Tuesday
4  26-03-2022    Saturday


In [21]:
print(X["distance_km"].describe())

print("\nMissing values:")
print(X.isna().sum().sort_values(ascending=False))

count    45593.000000
mean        99.303911
std       1099.731281
min          1.465067
25%          4.663493
50%          9.264281
75%         13.763977
max      19692.674606
Name: distance_km, dtype: float64

Missing values:
Delivery_person_Ratings        1908
Delivery_person_Age            1854
order_hour                     1731
Time_Orderd                    1731
City                           1200
multiple_deliveries             993
Weatherconditions               616
Road_traffic_density            601
Festival                        228
Delivery_person_ID                0
Restaurant_longitude              0
Restaurant_latitude               0
ID                                0
Delivery_location_latitude        0
Vehicle_condition                 0
Time_Order_picked                 0
Delivery_location_longitude       0
Order_Date                        0
Type_of_order                     0
Type_of_vehicle                   0
distance_km                       0
day_of_week      

In [22]:
print("\nData types:")
print(X.dtypes)

print("\nSample:")
display(X.head())


Data types:
ID                                 str
Delivery_person_ID                 str
Delivery_person_Age            float64
Delivery_person_Ratings        float64
Restaurant_latitude            float64
Restaurant_longitude           float64
Delivery_location_latitude     float64
Delivery_location_longitude    float64
Order_Date                         str
Time_Orderd                        str
Time_Order_picked                  str
Weatherconditions                  str
Road_traffic_density               str
Vehicle_condition                int64
Type_of_order                      str
Type_of_vehicle                    str
multiple_deliveries            float64
Festival                           str
City                               str
distance_km                    float64
order_hour                     float64
day_of_week                        str
dtype: object

Sample:


,ID,Delivery_person_ID,Delivery_person_Age,Delivery_person_Ratings,Restaurant_latitude,Restaurant_longitude,Delivery_location_latitude,Delivery_location_longitude,Order_Date,Time_Orderd,...,Road_traffic_density,Vehicle_condition,Type_of_order,Type_of_vehicle,multiple_deliveries,Festival,City,distance_km,order_hour,day_of_week
0,0x4607,INDORES13DEL02,37.0,4.9,22.745049,75.892471,22.765049,75.912471,19-03-2022,11:30:00,...,High,2,Snack,motorcycle,0.0,No,Urban,3.025149,11.50,Saturday
1,0xb379,BANGRES18DEL02,34.0,4.5,12.913041,77.683237,13.043041,77.813237,25-03-2022,19:45:00,...,Jam,2,Snack,scooter,1.0,No,Metropolitian,20.183530,19.75,Friday
2,0x5d6d,BANGRES19DEL01,23.0,4.4,12.914264,77.678400,12.924264,77.688400,19-03-2022,08:30:00,...,Low,0,Drinks,motorcycle,1.0,No,Urban,1.552758,8.50,Saturday
3,0x7a6a,COIMBRES13DEL02,38.0,4.7,11.003669,76.976494,11.053669,77.026494,05-04-2022,18:00:00,...,Medium,0,Buffet,motorcycle,1.0,No,Metropolitian,7.790401,18.00,Tuesday
4,0x70a2,CHENRES12DEL01,32.0,4.6,12.972793,80.249982,13.012793,80.289982,26-03-2022,13:30:00,...,High,1,Snack,scooter,1.0,No,Metropolitian,6.210138,13.50,Saturday


In [23]:
print("Feature matrix shape:", X.shape)

print("\nColumns:")
print(X.columns.tolist())

print("\nData types:")
print(X.dtypes)

print("\nMissing values:")
print(X.isna().sum().sort_values(ascending=False))

X.head()

Feature matrix shape: (45593, 22)

Columns:
['ID', 'Delivery_person_ID', 'Delivery_person_Age', 'Delivery_person_Ratings', 'Restaurant_latitude', 'Restaurant_longitude', 'Delivery_location_latitude', 'Delivery_location_longitude', 'Order_Date', 'Time_Orderd', 'Time_Order_picked', 'Weatherconditions', 'Road_traffic_density', 'Vehicle_condition', 'Type_of_order', 'Type_of_vehicle', 'multiple_deliveries', 'Festival', 'City', 'distance_km', 'order_hour', 'day_of_week']

Data types:
ID                                 str
Delivery_person_ID                 str
Delivery_person_Age            float64
Delivery_person_Ratings        float64
Restaurant_latitude            float64
Restaurant_longitude           float64
Delivery_location_latitude     float64
Delivery_location_longitude    float64
Order_Date                         str
Time_Orderd                        str
Time_Order_picked                  str
Weatherconditions                  str
Road_traffic_density               str
Vehicle_co

,ID,Delivery_person_ID,Delivery_person_Age,Delivery_person_Ratings,Restaurant_latitude,Restaurant_longitude,Delivery_location_latitude,Delivery_location_longitude,Order_Date,Time_Orderd,...,Road_traffic_density,Vehicle_condition,Type_of_order,Type_of_vehicle,multiple_deliveries,Festival,City,distance_km,order_hour,day_of_week
0,0x4607,INDORES13DEL02,37.0,4.9,22.745049,75.892471,22.765049,75.912471,19-03-2022,11:30:00,...,High,2,Snack,motorcycle,0.0,No,Urban,3.025149,11.50,Saturday
1,0xb379,BANGRES18DEL02,34.0,4.5,12.913041,77.683237,13.043041,77.813237,25-03-2022,19:45:00,...,Jam,2,Snack,scooter,1.0,No,Metropolitian,20.183530,19.75,Friday
2,0x5d6d,BANGRES19DEL01,23.0,4.4,12.914264,77.678400,12.924264,77.688400,19-03-2022,08:30:00,...,Low,0,Drinks,motorcycle,1.0,No,Urban,1.552758,8.50,Saturday
3,0x7a6a,COIMBRES13DEL02,38.0,4.7,11.003669,76.976494,11.053669,77.026494,05-04-2022,18:00:00,...,Medium,0,Buffet,motorcycle,1.0,No,Metropolitian,7.790401,18.00,Tuesday
4,0x70a2,CHENRES12DEL01,32.0,4.6,12.972793,80.249982,13.012793,80.289982,26-03-2022,13:30:00,...,High,1,Snack,scooter,1.0,No,Metropolitian,6.210138,13.50,Saturday


In [24]:
coord_cols = [
    "Restaurant_latitude",
    "Restaurant_longitude",
    "Delivery_location_latitude",
    "Delivery_location_longitude"
]

df[coord_cols].describe()

,Restaurant_latitude,Restaurant_longitude,Delivery_location_latitude,Delivery_location_longitude
count,45593.000000,45593.000000,45593.000000,45593.000000
mean,17.017729,70.231332,17.465186,70.845702
std,8.185109,22.883647,7.335122,21.118812
min,-30.905562,-88.366217,0.010000,0.010000
25%,12.933284,73.170000,12.988453,73.280000
50%,18.546947,75.898497,18.633934,76.002574
75%,22.728163,78.044095,22.785049,78.107044
max,30.914057,88.433452,31.054057,88.563452


In [25]:
zero_restaurant_coords = (
    (df["Restaurant_latitude"] == 0)
    & (df["Restaurant_longitude"] == 0)
)

print("Rows with restaurant coordinates (0, 0):",
      zero_restaurant_coords.sum())

print(
    "Percentage:",
    round(zero_restaurant_coords.mean() * 100, 2),
    "%"
)

print("\nNegative restaurant latitudes:",
      (df["Restaurant_latitude"] < 0).sum())

print("Negative restaurant longitudes:",
      (df["Restaurant_longitude"] < 0).sum())

print("Negative delivery latitudes:",
      (df["Delivery_location_latitude"] < 0).sum())

print("Negative delivery longitudes:",
      (df["Delivery_location_longitude"] < 0).sum())

Rows with restaurant coordinates (0, 0): 3640
Percentage: 7.98 %

Negative restaurant latitudes: 431
Negative restaurant longitudes: 162
Negative delivery latitudes: 0
Negative delivery longitudes: 0


In [26]:
coordinate_check = df[coord_cols].copy()
coordinate_check["distance_km"] = X["distance_km"]

coordinate_check.sort_values(
    "distance_km",
    ascending=False
).head(20)

,Restaurant_latitude,Restaurant_longitude,Delivery_location_latitude,Delivery_location_longitude,distance_km
33533,-22.533662,-88.366217,22.663662,88.496217,19692.674606
6788,-22.552672,-88.352885,22.662672,88.462885,19688.001288
2484,-22.538999,-88.322337,22.668999,88.452337,19683.687561
18826,-22.537960,-88.349843,22.547960,88.359843,19677.180552
9535,-23.374989,-85.335486,23.444989,85.405486,19070.408110
762,-23.359194,-85.325447,23.449194,85.415447,19070.337839
35535,-23.369746,-85.339820,23.419746,85.389820,19069.158946
22291,-23.351058,-85.325731,23.421058,85.395731,19068.246962
30633,-23.357804,-85.325146,23.417804,85.385146,19067.128547
43454,-23.351058,-85.325731,23.401058,85.375731,19066.150742


### Coordinate Quality Handling

Inspection of the raw coordinates has revealed two data-quality issues.

Some restaurant coordinates contained negative latitude or longitude values while the corresponding delivery coordinates were positive and geographically close in absolute terms. These negative signs produced impossible delivery distances of thousands of kilometres. The restaurant coordinates will therefore be converted to their absolute values before calculating distance.

Additionally, some restaurant locations were recorded as `(0, 0)`. These were treated as missing coordinates rather than valid geographical locations. Their resulting `distance_km` values will be left missing for later imputation within the preprocessing pipeline.

In [27]:
restaurant_lat = df["Restaurant_latitude"].abs().copy()
restaurant_lon = df["Restaurant_longitude"].abs().copy()

delivery_lat = df["Delivery_location_latitude"].copy()
delivery_lon = df["Delivery_location_longitude"].copy()

invalid_restaurant_coords = (
    (restaurant_lat == 0)
    & (restaurant_lon == 0)
)

restaurant_lat.loc[invalid_restaurant_coords] = np.nan
restaurant_lon.loc[invalid_restaurant_coords] = np.nan

X["distance_km"] = haversine_distance(
    restaurant_lat,
    restaurant_lon,
    delivery_lat,
    delivery_lon
)

print(X["distance_km"].describe())

print(
    "\nMissing distances:",
    X["distance_km"].isna().sum()
)

count    41953.000000
mean         9.720284
std          5.603083
min          1.465067
25%          4.657655
50%          9.193021
75%         13.680920
max         20.969489
Name: distance_km, dtype: float64

Missing distances: 3640


## Train-Test Split

The dataset is split before fitting any learned preprocessing steps. This prevents information from the test set from influencing imputation, scaling, category learning, or model training.

An 80/20 split is used with a fixed random state for reproducibility.

In [28]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Training features:", X_train.shape)
print("Test features:", X_test.shape)
print("Training target:", y_train.shape)
print("Test target:", y_test.shape)

Training features: (36474, 22)
Test features: (9119, 22)
Training target: (36474,)
Test target: (9119,)


In [29]:
numerical_features = [
    "Delivery_person_Age",
    "Delivery_person_Ratings",
    "Vehicle_condition",
    "multiple_deliveries",
    "distance_km",
    "order_hour"
]

categorical_features = [
    "Weatherconditions",
    "Road_traffic_density",
    "Type_of_order",
    "Type_of_vehicle",
    "Festival",
    "City",
    "day_of_week"
]

print("Numerical features:", len(numerical_features))
print("Categorical features:", len(categorical_features))
print("Total:", len(numerical_features) + len(categorical_features))

Numerical features: 6
Categorical features: 7
Total: 13


In [30]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

numerical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            )
        )
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            numerical_pipeline,
            numerical_features
        ),
        (
            "cat",
            categorical_pipeline,
            categorical_features
        )
    ]
)


## Baseline Model

A simple mean-prediction baseline is established before training the machine-learning models. This will provide a reference point for determining whether the learned models provide useful predictive improvement.

In [31]:
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

baseline = DummyRegressor(strategy="mean")

baseline.fit(X_train, y_train)

baseline_predictions = baseline.predict(X_test)

baseline_mae = mean_absolute_error(
    y_test,
    baseline_predictions
)

baseline_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        baseline_predictions
    )
)

baseline_r2 = r2_score(
    y_test,
    baseline_predictions
)

print(f"Baseline MAE:  {baseline_mae:.2f} minutes")
print(f"Baseline RMSE: {baseline_rmse:.2f} minutes")
print(f"Baseline R²:   {baseline_r2:.3f}")

Baseline MAE:  7.58 minutes
Baseline RMSE: 9.36 minutes
Baseline R²:   -0.000


## Regression Model Comparison

Will compare three regression algorithms:

1. **Ridge Regression** — provides a regularised linear benchmark.
2. **Random Forest Regressor** — captures nonlinear relationships and interactions through an ensemble of decision trees.
3. **Gradient Boosting Regressor** — builds trees sequentially, with later trees attempting to correct errors made by earlier trees.

Each model will use the same feature set and leakage-safe preprocessing architecture.

In [32]:
from sklearn.linear_model import Ridge

ridge_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", Ridge())
    ]
)

ridge_pipeline.fit(X_train, y_train)

ridge_predictions = ridge_pipeline.predict(X_test)

ridge_mae = mean_absolute_error(
    y_test,
    ridge_predictions
)

ridge_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        ridge_predictions
    )
)

ridge_r2 = r2_score(
    y_test,
    ridge_predictions
)

print(f"Ridge MAE:  {ridge_mae:.2f} minutes")
print(f"Ridge RMSE: {ridge_rmse:.2f} minutes")
print(f"Ridge R²:   {ridge_r2:.3f}")

Ridge MAE:  4.80 minutes
Ridge RMSE: 6.05 minutes
Ridge R²:   0.583


In [33]:
from sklearn.ensemble import RandomForestRegressor

random_forest_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            RandomForestRegressor(
                n_estimators=100,
                random_state=42,
                n_jobs=-1
            )
        )
    ]
)

random_forest_pipeline.fit(X_train, y_train)

rf_predictions = random_forest_pipeline.predict(X_test)

rf_mae = mean_absolute_error(
    y_test,
    rf_predictions
)

rf_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        rf_predictions
    )
)

rf_r2 = r2_score(
    y_test,
    rf_predictions
)

print(f"Random Forest MAE:  {rf_mae:.2f} minutes")
print(f"Random Forest RMSE: {rf_rmse:.2f} minutes")
print(f"Random Forest R²:   {rf_r2:.3f}")

Random Forest MAE:  3.30 minutes
Random Forest RMSE: 4.17 minutes
Random Forest R²:   0.802


In [34]:
from sklearn.ensemble import GradientBoostingRegressor

gradient_boosting_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            GradientBoostingRegressor(
                random_state=42
            )
        )
    ]
)

gradient_boosting_pipeline.fit(
    X_train,
    y_train
)

gb_predictions = gradient_boosting_pipeline.predict(
    X_test
)

gb_mae = mean_absolute_error(
    y_test,
    gb_predictions
)

gb_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        gb_predictions
    )
)

gb_r2 = r2_score(
    y_test,
    gb_predictions
)

print(f"Gradient Boosting MAE:  {gb_mae:.2f} minutes")
print(f"Gradient Boosting RMSE: {gb_rmse:.2f} minutes")
print(f"Gradient Boosting R²:   {gb_r2:.3f}")

Gradient Boosting MAE:  3.66 minutes
Gradient Boosting RMSE: 4.60 minutes
Gradient Boosting R²:   0.759


In [35]:
results = pd.DataFrame({
    "Model": [
        "Baseline",
        "Ridge Regression",
        "Random Forest",
        "Gradient Boosting"
    ],
    "MAE": [
        baseline_mae,
        ridge_mae,
        rf_mae,
        gb_mae
    ],
    "RMSE": [
        baseline_rmse,
        ridge_rmse,
        rf_rmse,
        gb_rmse
    ],
    "R²": [
        baseline_r2,
        ridge_r2,
        rf_r2,
        gb_r2
    ]
})

results = results.sort_values("MAE")

results.round(3)

,Model,MAE,RMSE,R²
2,Random Forest,3.300,4.169,0.802
3,Gradient Boosting,3.663,4.600,0.759
1,Ridge Regression,4.803,6.045,0.583
0,Baseline,7.579,9.364,-0.000


## Hyperparameter Tuning

Random Forest produced the strongest initial performance, so it is selected for hyperparameter tuning.

I will use `RandomizedSearchCV` with cross-validation on the training set. The test set is not used for hyperparameter selection, preserving it for final evaluation.

The tuning process explores the number and depth of trees, feature sampling, and minimum sample requirements for splitting and leaf nodes.

In [36]:
from sklearn.model_selection import RandomizedSearchCV

rf_param_distributions = {
    "model__n_estimators": [100, 200, 300],
    "model__max_depth": [None, 10, 20, 30],
    "model__min_samples_split": [2, 5, 10],
    "model__min_samples_leaf": [1, 2, 4],
    "model__max_features": ["sqrt", 0.7, 1.0]
}

rf_search = RandomizedSearchCV(
    estimator=random_forest_pipeline,
    param_distributions=rf_param_distributions,
    n_iter=15,
    scoring="neg_mean_absolute_error",
    cv=3,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

rf_search.fit(X_train, y_train)

Fitting 3 folds for each of 15 candidates, totalling 45 fits


,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'model__max_depth': [None, 10, ...], 'model__max_features': ['sqrt', 0.7, ...], 'model__min_samples_leaf': [1, 2, ...], 'model__min_samples_split': [2, 5, ...], ...}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",15
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'neg_mean_absolute_error'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``RandomizedSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variou

In [37]:
print("Best parameters:")
print(rf_search.best_params_)

print(
    "\nBest cross-validation MAE:",
    round(-rf_search.best_score_, 3),
    "minutes"
)

Best parameters:
{'model__n_estimators': 200, 'model__min_samples_split': 10, 'model__min_samples_leaf': 1, 'model__max_features': 0.7, 'model__max_depth': 20}

Best cross-validation MAE: 3.224 minutes


In [38]:
best_rf_model = rf_search.best_estimator_

tuned_rf_predictions = best_rf_model.predict(X_test)

tuned_rf_mae = mean_absolute_error(
    y_test,
    tuned_rf_predictions
)

tuned_rf_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        tuned_rf_predictions
    )
)

tuned_rf_r2 = r2_score(
    y_test,
    tuned_rf_predictions
)

print(f"Tuned Random Forest MAE:  {tuned_rf_mae:.3f} minutes")
print(f"Tuned Random Forest RMSE: {tuned_rf_rmse:.3f} minutes")
print(f"Tuned Random Forest R²:   {tuned_rf_r2:.3f}")

rf_comparison = pd.DataFrame({
    "Model": [
        "Default Random Forest",
        "Tuned Random Forest"
    ],
    "MAE": [
        rf_mae,
        tuned_rf_mae
    ],
    "RMSE": [
        rf_rmse,
        tuned_rf_rmse
    ],
    "R²": [
        rf_r2,
        tuned_rf_r2
    ]
})

rf_comparison.round(3)

Tuned Random Forest MAE:  3.240 minutes
Tuned Random Forest RMSE: 4.072 minutes
Tuned Random Forest R²:   0.811


,Model,MAE,RMSE,R²
0,Default Random Forest,3.30,4.169,0.802
1,Tuned Random Forest,3.24,4.072,0.811


In [39]:
error_analysis = X_test.copy()

error_analysis["actual_eta"] = y_test
error_analysis["predicted_eta"] = tuned_rf_predictions

error_analysis["error"] = (
    error_analysis["actual_eta"]
    - error_analysis["predicted_eta"]
)

error_analysis["absolute_error"] = (
    error_analysis["error"].abs()
)

error_analysis.head()

,ID,Delivery_person_ID,Delivery_person_Age,Delivery_person_Ratings,Restaurant_latitude,Restaurant_longitude,Delivery_location_latitude,Delivery_location_longitude,Order_Date,Time_Orderd,...,multiple_deliveries,Festival,City,distance_km,order_hour,day_of_week,actual_eta,predicted_eta,error,absolute_error
7725,0x38f7,INDORES16DEL02,30.0,5.0,22.744648,75.894377,22.824648,75.974377,20-03-2022,18:20:00,...,0.0,Yes,Metropolitian,12.099391,18.333333,Sunday,44.0,43.240389,0.759611,0.759611
949,0x1cf1,SURRES08DEL03,31.0,4.7,21.173343,72.792731,21.263343,72.882731,31-03-2022,19:10:00,...,2.0,No,Metropolitian,13.681492,19.166667,Thursday,41.0,41.540018,-0.540018,0.540018
44893,0x2838,COIMBRES17DEL03,35.0,3.6,11.026117,76.944652,11.116117,77.034652,29-03-2022,21:25:00,...,0.0,No,Metropolitian,14.021727,21.416667,Tuesday,39.0,35.132046,3.867954,3.867954
42071,0x49e0,CHENRES04DEL01,35.0,4.6,13.029780,80.208812,13.139780,80.318812,18-03-2022,17:40:00,...,2.0,No,Metropolitian,17.074786,17.666667,Friday,38.0,40.612916,-2.612916,2.612916
27091,0x9c4,RANCHIRES07DEL01,20.0,4.6,23.359407,85.325055,23.429407,85.395055,29-03-2022,21:45:00,...,NaN,No,Metropolitian,10.564974,21.750000,Tuesday,38.0,42.463022,-4.463022,4.463022


In [40]:
error_analysis.sort_values(
    "absolute_error",
    ascending=False
).head(15)

,ID,Delivery_person_ID,Delivery_person_Age,Delivery_person_Ratings,Restaurant_latitude,Restaurant_longitude,Delivery_location_latitude,Delivery_location_longitude,Order_Date,Time_Orderd,...,multiple_deliveries,Festival,City,distance_km,order_hour,day_of_week,actual_eta,predicted_eta,error,absolute_error
44369,0x11ad,VADRES17DEL02,37.0,4.7,0.000000,0.000000,0.080000,0.080000,25-03-2022,17:20:00,...,1.0,No,Metropolitian,NaN,17.333333,Friday,44.0,21.531907,22.468093,22.468093
39858,0x443,COIMBRES08DEL03,NaN,NaN,-11.001852,-76.976268,11.091852,77.066268,31-03-2022,NaN,...,1.0,No,Metropolitian,14.022296,NaN,Thursday,54.0,32.289868,21.710132,21.710132
30893,0x2967,MUMRES04DEL01,NaN,NaN,19.254567,72.848923,19.294567,72.888923,24-03-2022,NaN,...,1.0,No,Metropolitian,6.116387,NaN,Thursday,53.0,32.441920,20.558080,20.558080
3510,0x542,JAPRES18DEL02,NaN,NaN,-26.913987,75.752891,27.043987,75.882891,23-03-2022,NaN,...,1.0,No,Metropolitian,19.362542,NaN,Wednesday,12.0,31.449266,-19.449266,19.449266
14829,0xdb20,AURGRES16DEL01,38.0,4.8,0.000000,0.000000,0.070000,0.070000,14-02-2022,20:30:00,...,1.0,No,Metropolitian,NaN,20.500000,Monday,53.0,34.579549,18.420451,18.420451
18387,0x6efa,VADRES08DEL03,37.0,4.7,0.000000,0.000000,0.090000,0.090000,27-03-2022,23:25:00,...,NaN,No,Metropolitian,NaN,23.416667,Sunday,44.0,26.435602,17.564398,17.564398
30538,0x8fb,JAPRES04DEL03,NaN,NaN,26.902328,75.794257,27.042328,75.934257,27-03-2022,NaN,...,1.0,No,Metropolitian,20.852514,NaN,Sunday,20.0,37.433431,-17.433431,17.433431
11427,0x8d9,RANCHIRES11DEL02,NaN,NaN,23.359194,85.325447,23.439194,85.405447,29-03-2022,NaN,...,NaN,No,Urban,12.074056,NaN,Tuesday,40.0,22.663235,17.336765,17.336765
37078,0x64e2,VADRES18DEL01,36.0,4.6,0.000000,0.000000,0.070000,0.070000,08-03-2022,20:00:00,...,1.0,No,Semi-Urban,NaN,20.000000,Tuesday,54.0,36.705043,17.294957,17.294957
31768,0xcf68,GOARES05DEL01,31.0,4.6,0.000000,0.000000,0.070000,0.070000,12-02-2022,21:25:00,...,0.0,No,Metropolitian,NaN,21.416667,Saturday,48.0,31.145274,16.854726,16.854726


In [41]:
traffic_errors = (
    error_analysis
    .groupby("Road_traffic_density", dropna=False)
    ["absolute_error"]
    .agg(["mean", "median", "count"])
    .sort_values("mean", ascending=False)
)

traffic_errors



,mean,median,count
Road_traffic_density,,,
NaN,6.354015,6.170091,127
Jam,3.679700,3.222976,2826
High,3.389757,2.802121,864
Medium,3.113574,2.692140,2179
Low,2.761229,2.436561,3123


In [42]:
weather_errors = (
    error_analysis
    .groupby("Weatherconditions", dropna=False)
    ["absolute_error"]
    .agg(["mean", "median", "count"])
    .sort_values("mean", ascending=False)
)

weather_errors


,mean,median,count
Weatherconditions,,,
NaN,6.354015,6.170091,127
Sandstorms,3.372379,2.880393,1568
Stormy,3.357122,2.881909,1567
Windy,3.354898,3.028625,1434
Sunny,3.137902,2.893272,1450
Cloudy,3.000396,2.475748,1476
Fog,2.937483,2.462276,1497


In [43]:
city_errors = (
    error_analysis
    .groupby("City", dropna=False)
    ["absolute_error"]
    .agg(["mean", "median", "count"])
    .sort_values("mean", ascending=False)
)

city_errors

,mean,median,count
City,,,
Semi-Urban,4.004030,3.034621,27
Urban,3.247691,2.837560,2026
Metropolitian,3.238295,2.753389,6811
NaN,3.130909,2.912562,255


In [44]:
error_analysis["distance_group"] = pd.cut(
    error_analysis["distance_km"],
    bins=[0, 5, 10, 15, 25],
    labels=[
        "0–5 km",
        "5–10 km",
        "10–15 km",
        "15+ km"
    ]
)

distance_errors = (
    error_analysis
    .groupby(
        "distance_group",
        observed=True
    )["absolute_error"]
    .agg(["mean", "median", "count"])
)

distance_errors

,mean,median,count
distance_group,,,
0–5 km,2.941406,2.502039,2179
5–10 km,3.117838,2.543606,2288
10–15 km,3.438042,3.084072,2276
15+ km,3.362779,2.923295,1633


In [45]:
error_analysis["missing_feature_count"] = (
    X_test.isna().sum(axis=1)
)

missing_error_analysis = (
    error_analysis
    .groupby("missing_feature_count")
    ["absolute_error"]
    .agg(["mean", "median", "count"])
)

missing_error_analysis


,mean,median,count
missing_feature_count,,,
0,3.102565,2.704157,7581
1,3.420151,2.979986,1094
2,4.243882,3.509124,86
3,3.408429,3.524151,5
4,4.865412,3.929054,212
5,3.910983,2.839665,29
6,6.655634,6.772372,99
7,7.256837,5.107443,13


In [46]:
error_analysis["has_missing_features"] = (
    X_test.isna().any(axis=1)
)

missing_comparison = (
    error_analysis
    .groupby("has_missing_features")
    ["absolute_error"]
    .agg(["mean", "median", "count"])
)

missing_comparison

,mean,median,count
has_missing_features,,,
False,3.102565,2.704157,7581
True,3.915341,3.236795,1538


In [47]:
print("TRAFFIC")
display(traffic_errors)

print("\nWEATHER")
display(weather_errors)

print("\nCITY")
display(city_errors)

print("\nDISTANCE")
display(distance_errors)

print("\nMISSING FEATURES")
display(missing_error_analysis)

print("\nCOMPLETE VS INCOMPLETE ROWS")
display(missing_comparison)

TRAFFIC


,mean,median,count
Road_traffic_density,,,
NaN,6.354015,6.170091,127
Jam,3.679700,3.222976,2826
High,3.389757,2.802121,864
Medium,3.113574,2.692140,2179
Low,2.761229,2.436561,3123



WEATHER


,mean,median,count
Weatherconditions,,,
NaN,6.354015,6.170091,127
Sandstorms,3.372379,2.880393,1568
Stormy,3.357122,2.881909,1567
Windy,3.354898,3.028625,1434
Sunny,3.137902,2.893272,1450
Cloudy,3.000396,2.475748,1476
Fog,2.937483,2.462276,1497



CITY


...

,mean,median,count
distance_group,,,
0–5 km,2.941406,2.502039,2179
5–10 km,3.117838,2.543606,2288
10–15 km,3.438042,3.084072,2276
15+ km,3.362779,2.923295,1633



MISSING FEATURES


,mean,median,count
missing_feature_count,,,
0,3.102565,2.704157,7581
1,3.420151,2.979986,1094
2,4.243882,3.509124,86
3,3.408429,3.524151,5
4,4.865412,3.929054,212
5,3.910983,2.839665,29
6,6.655634,6.772372,99
7,7.256837,5.107443,13



COMPLETE VS INCOMPLETE ROWS


,mean,median,count
has_missing_features,,,
False,3.102565,2.704157,7581
True,3.915341,3.236795,1538


In [48]:
feature_names = (
    best_rf_model
    .named_steps["preprocessor"]
    .get_feature_names_out()
)

feature_importances = (
    best_rf_model
    .named_steps["model"]
    .feature_importances_
)

importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": feature_importances
}).sort_values(
    "importance",
    ascending=False
)

importance_df.head(15)

,feature,importance
1,num__Delivery_person_Ratings,0.207722
14,cat__Road_traffic_density_Low,0.115024
3,num__multiple_deliveries,0.112918
4,num__distance_km,0.108308
0,num__Delivery_person_Age,0.095803
2,num__Vehicle_condition,0.076294
10,cat__Weatherconditions_Sunny,0.069689
7,cat__Weatherconditions_Fog,0.040374
6,cat__Weatherconditions_Cloudy,0.039943
5,num__order_hour,0.036105


### Error Analysis Findings

The tuned Random Forest achieved an overall test MAE of 3.24 minutes, but performance was not uniform across all orders.

Prediction error was higher under heavier traffic conditions. Among orders with known traffic information, low-traffic orders had the lowest MAE (2.76 minutes), while orders classified as traffic jams had the highest MAE (3.68 minutes). Orders with missing traffic information had a substantially higher MAE of 6.35 minutes.

Weather-related differences were smaller. MAE across known weather categories ranged from approximately 2.94 to 3.37 minutes. Missing weather information was again associated with substantially higher error.

Distance also affected performance. Orders within 0–5 km had an MAE of 2.94 minutes, compared with 3.44 minutes for orders between 10–15 km. However, error did not increase monotonically across all distance groups.

Semi-urban orders had a higher observed MAE, but only 27 semi-urban examples were present in the test set, so this result should not be interpreted as strong evidence of systematically worse performance in semi-urban locations.

Missing data was one of the clearest weaknesses. Complete test rows had an MAE of 3.10 minutes, compared with 3.92 minutes for rows containing at least one missing feature. Error generally increased as more features were missing, although some groups contained relatively few observations.

This suggests that while pipeline-based imputation allows the model to make predictions when information is unavailable, missing operational information still reduces predictive performance.